## Logit Adjusted Loss

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class LogitAdjustedLoss(nn.Module):
    def __init__(self, cls_num_list, tau=1.0):
        super(LogitAdjustedLoss, self).__init__()
        """
        Args:
            cls_num_list: List of integers containing the number of samples per class.
            tau: Hyperparameter to scale the adjustment (default=1.0).
        """
        cls_num_list = torch.FloatTensor(cls_num_list)
        cls_priors = cls_num_list / cls_num_list.sum()
        
        # Calculate the log-prior offset: tau * ln(pi_i)
        # We use a small epsilon to avoid log(0)
        self.logit_adj = tau * torch.log(cls_priors + 1e-12)
        self.logit_adj = self.logit_adj.view(1, -1) # Reshape for broadcasting

    def forward(self, logits, target):
        """
        Args:
            logits: Prediction scores from the model (Batch, Classes)
            target: Ground truth labels (Batch)
        """
        # Move adjustment to the same device as logits
        adjustment = self.logit_adj.to(logits.device)
        
        # Apply adjustment: z_i_adj = f_yi(x) + logit_adjustment
        # Note: In the paper's loss formulation, we ADD the ln(pi) term 
        # to the logits inside the Softmax to enforce the margin.
        adjusted_logits = logits + adjustment
        
        return F.cross_entropy(adjusted_logits, target)

# --- Usage Example ---
# Assume 3 classes with 1000, 100, and 10 samples respectively
class_counts = [1000, 100, 10]
criterion = LogitAdjustedLoss(cls_num_list=class_counts, tau=1.0)

# Mock model output (logits) and labels
mock_logits = torch.randn(8, 3) # Batch of 8, 3 classes
mock_labels = torch.tensor([0, 0, 1, 1, 2, 2, 0, 1])

loss = criterion(mock_logits, mock_labels)
print(f"Logit Adjusted Loss: {loss.item():.4f}")

Logit Adjusted Loss: 2.0001
